# Comparaison -- pipeline complet vs baselines (direct llm / cot)

Compare, sur les MEMES 5 modeles x 3 strategies x 500 scenarios, trois
approches :
- **pipeline** -- le graphe complet (intent_grounding -> necessity_checker
  -> decision_engine CP-SAT -> explanation), voir eval/scoring.ipynb.
- **direct** -- un seul appel LLM qui fait extraction + decision + placement
  en une fois, sans solveur dedie.
- **cot** -- la meme chose, avec une procedure en 6 etapes explicitee dans le
  prompt.

Les trois sont scores EXACTEMENT de la meme facon : la decision STAY/OFFLOAD/
INFEASIBLE comparee a `ground_truth.decision`, et la validite REELLE du
placement final verifiee contre les VRAIES exigences du scenario avec
`check_candidate_placement` -- jamais contre ce que le modele a lui-meme
extrait/propose comme etant correct.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent  # repo root, assuming this notebook runs from eval/
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "eval" / "system_model"))

from agent.model.schemas import Requirement  # noqa: E402
from agent.utils.pipeline import check_candidate_placement  # noqa: E402
from loader import load_flows, load_links, load_nodes, load_services  # noqa: E402

PIPELINE_RESULTS_DIR = ROOT / "eval" / "results"
BASELINE_RESULTS_DIR = ROOT / "eval" / "baselines" / "results"
KPI_REMAP = {"e2e_latency": "latency"}

MODEL_SHORT_NAME = {
    "google/gemini-2.5-flash-lite": "gemini-2.5-flash-lite",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral-small-3.2-24b",
    "openai/gpt-oss-20b": "gpt-oss-20b",
    "meta-llama/llama-3.3-70b-instruct": "llama-3.3-70b",
    "openai/gpt-oss-120b": "gpt-oss-120b",
}
MODEL_ORDER = ["gemini-2.5-flash-lite", "gpt-oss-120b", "gpt-oss-20b", "llama-3.3-70b", "mistral-small-3.2-24b"]
STRATEGY_ORDER = ["zero_shot", "one_shot", "few_shot"]
APPROACH_ORDER = ["pipeline", "ablation", "direct", "cot"]
APPROACH_COLORS = {"pipeline": "#3569A8", "ablation": "#28A69A", "direct": "#E99A43", "cot": "#8D6A9F"}
APPROACH_LABELS = {"pipeline": "Pipeline complet", "ablation": "Ablation (sans necessity_checker)",
                    "direct": "Direct LLM", "cot": "CoT"}


## Fixtures partagees (nodes/links/vraies exigences/placement de depart, par scenario)

In [ ]:
with open(ROOT / "eval" / "experiment_500" / "scenarios.json") as f:
    SCENARIOS_BY_ID = {s["id"]: s for s in json.load(f)}

NODES = load_nodes()
LINKS = load_links()
_fixtures_cache = {}


def scenario_fixtures(scenario_id):
    if scenario_id not in _fixtures_cache:
        scenario = SCENARIOS_BY_ID[scenario_id]
        services = load_services(scenario)
        flows = load_flows(scenario)
        gt_reqs = []
        for i, req in enumerate(scenario["requirements"]):
            kpi = KPI_REMAP.get(req["kpi_type"], req["kpi_type"])
            gt_reqs.append(Requirement(
                kpi_type=kpi, comparator=req["comparator"], target_value=req["new_value"],
                unit=req["unit"], target_type=req["target_type"], target_id=req["target_id"],
                source_span="", requirement_id=f"gt-{i}",
            ))
        baseline_placement = {sid: s.current_node for sid, s in services.items()}
        _fixtures_cache[scenario_id] = (services, flows, gt_reqs, baseline_placement)
    return _fixtures_cache[scenario_id]


def full_placement(predicted_placement, baseline_placement):
    if predicted_placement is None:
        return dict(baseline_placement)
    return {sid: predicted_placement.get(sid, node) for sid, node in baseline_placement.items()}


## Chargement -- pipeline (eval/results/*.jsonl)

Memes champs/logique que eval/scoring.ipynb : `necessity_result`/
`decision_result` -> label STAY/OFFLOAD/INFEASIBLE, `decision_result.
new_configuration` comme placement final (ou `necessity_result.
current_placement` si necessity_checker a court-circuite).


In [ ]:
PIPELINE_FILES = [
    "google-gemini-2.5-flash-lite__zero_shot__20261103T220001Z.jsonl",
    "google-gemini-2.5-flash-lite__one_shot__20261103T220001Z.jsonl",
    "google-gemini-2.5-flash-lite__few_shot__20261103T220001Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__zero_shot__20261103T220001Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__one_shot__20261103T220001Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__few_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-20b__zero_shot__20261103T220000Z.jsonl",
    "openai-gpt-oss-20b__one_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-20b__few_shot__20261103T220001Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__zero_shot__20261103T220001Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__one_shot__20261103T220001Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__few_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-120b__zero_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-120b__one_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-120b__few_shot__20261103T220001Z.jsonl",
]

ABLATION_FILES = [
    "google-gemini-2.5-flash-lite__zero_shot__20261104T121659Z.jsonl",
    "google-gemini-2.5-flash-lite__one_shot__20261104T121703Z.jsonl",
    "google-gemini-2.5-flash-lite__few_shot__20261104T121705Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__zero_shot__20261104T121709Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__one_shot__20261104T121712Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__few_shot__20261104T121715Z.jsonl",
    "openai-gpt-oss-20b__zero_shot__20261104T121718Z.jsonl",
    "openai-gpt-oss-20b__one_shot__20261104T121721Z.jsonl",
    "openai-gpt-oss-20b__few_shot__20261104T121724Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__zero_shot__20261104T121727Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__one_shot__20261104T121730Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__few_shot__20261104T121733Z.jsonl",
    "openai-gpt-oss-120b__zero_shot__20261104T121737Z.jsonl",
    "openai-gpt-oss-120b__one_shot__20261104T121740Z.jsonl",
    "openai-gpt-oss-120b__few_shot__20261104T121743Z.jsonl",
]
ABLATION_RESULTS_DIR = ROOT / "eval" / "results_ablation"


def pipeline_predicted_decision(record):
    necessity = record["necessity_result"]
    decision = record["decision_result"]
    if necessity is not None and necessity["status"] == "reconfiguration_not_required":
        return "STAY"
    if decision is None:
        return "STAY"
    if decision["outcome"] == "INFEASIBLE":
        return "INFEASIBLE"
    if not decision["decision"]:
        return "STAY"
    return "OFFLOAD"


def pipeline_final_configuration(record):
    if record["decision_result"] is not None:
        return record["decision_result"]["new_configuration"]
    return record["necessity_result"]["current_placement"]


def total_token_usage(record):
    """Sum of both LLM calls' total_tokens -- extraction + explanation,
    the two real API calls the pipeline/ablation make per scenario."""
    total = 0
    for key in ("extraction_token_usage", "explanation_token_usage"):
        usage = record.get(key)
        if usage:
            total += usage.get("total_tokens", 0)
    return total


def load_pipeline_like(results_dir, files, approach_name):
    """Shared loader for eval/results/ (pipeline) and eval/results_ablation/
    (ablation) -- identical record shape (run_sample_full_graph.py:run),
    only skip_necessity_checker differs at generation time."""
    rows = []
    for name in files:
        with open(results_dir / name) as fh:
            for line in fh:
                r = json.loads(line)
                if "error" in r:
                    continue
                rc = r["run_config"]
                services, flows, gt_reqs, baseline_placement = scenario_fixtures(r["scenario_id"])

                pred_decision = pipeline_predicted_decision(r)
                cfg = pipeline_final_configuration(r)
                pred_moved = frozenset(sid for sid, node in cfg.items() if node != baseline_placement[sid])
                gt_moved = frozenset(f"C{n}" for n in r["ground_truth"]["moved_components"])
                gt_decision = r["ground_truth"]["decision"]

                valid, _checks = check_candidate_placement(cfg, services, NODES, LINKS, gt_reqs, flows)

                rows.append({
                    "approach": approach_name,
                    "model": rc["model"], "model_short": MODEL_SHORT_NAME[rc["model"]],
                    "strategy": rc["prompt_strategy"], "scenario_id": r["scenario_id"],
                    "parsed_ok": True,
                    "pred_decision": pred_decision, "gt_decision": gt_decision,
                    "decision_correct": pred_decision == gt_decision,
                    "n_moved_pred": len(pred_moved), "n_moved_gt": len(gt_moved),
                    "moved_correct": (pred_moved == gt_moved) if gt_decision == "OFFLOAD" else None,
                    "real_valid": valid,
                    "elapsed_s": r.get("total_elapsed_s"),
                    "total_tokens": total_token_usage(r),
                })
    return pd.DataFrame(rows)


df_pipeline = load_pipeline_like(PIPELINE_RESULTS_DIR, PIPELINE_FILES, "pipeline")
df_ablation = load_pipeline_like(ABLATION_RESULTS_DIR, ABLATION_FILES, "ablation")
df_pipeline.shape, df_ablation.shape

## Chargement -- baselines (eval/baselines/results/*.jsonl), meme logique que eval/baselines/scoring.ipynb

In [ ]:
rows = []
for f in sorted(BASELINE_RESULTS_DIR.glob("*.jsonl")):
    with open(f) as fh:
        for line in fh:
            r = json.loads(line)
            if "error" in r:
                continue
            rc = r["run_config"]
            result = r.get("result")
            services, flows, gt_reqs, baseline_placement = scenario_fixtures(r["scenario_id"])

            parsed_ok = result is not None
            pred_decision = result.get("decision") if result else None
            pred_full_placement = full_placement(result.get("placement") if result else None, baseline_placement)
            pred_moved = frozenset(sid for sid, node in pred_full_placement.items() if node != baseline_placement[sid])
            gt_moved = frozenset(f"C{n}" for n in r["ground_truth"]["moved_components"])
            gt_decision = r["ground_truth"]["decision"]

            candidate = pred_full_placement if parsed_ok else baseline_placement
            valid, _checks = check_candidate_placement(candidate, services, NODES, LINKS, gt_reqs, flows)

            usage = r.get("token_usage") or {}

            rows.append({
                "approach": rc["baseline"],
                "model": rc["model"], "model_short": MODEL_SHORT_NAME[rc["model"]],
                "strategy": rc["prompt_strategy"], "scenario_id": r["scenario_id"],
                "parsed_ok": parsed_ok,
                "pred_decision": pred_decision, "gt_decision": gt_decision,
                "decision_correct": (pred_decision == gt_decision) if parsed_ok else False,
                "n_moved_pred": len(pred_moved) if parsed_ok else None,
                "n_moved_gt": len(gt_moved),
                "moved_correct": (pred_moved == gt_moved) if (parsed_ok and gt_decision == "OFFLOAD") else None,
                "real_valid": valid if parsed_ok else False,
                "elapsed_s": r.get("elapsed_s"),
                "total_tokens": usage.get("total_tokens"),
            })

df_baselines = pd.DataFrame(rows)
df = pd.concat([df_pipeline, df_ablation, df_baselines], ignore_index=True)
df.shape

## Tableau comparatif -- exactitude de la decision, par approche x modele (strategies confondues)

In [ ]:
decision_acc = df.groupby(["approach", "model_short"])["decision_correct"].mean().unstack("approach")
display(decision_acc.reindex(index=MODEL_ORDER)[APPROACH_ORDER].round(3))


## Tableau comparatif -- validite REELLE du placement (STAY + OFFLOAD, exclut INFEASIBLE)

In [ ]:
real_valid = df[df["gt_decision"] != "INFEASIBLE"].groupby(["approach", "model_short"])["real_valid"].mean().unstack("approach")
display(real_valid.reindex(index=MODEL_ORDER)[APPROACH_ORDER].round(3))


## Tableau comparatif -- exactitude de l'ensemble des services deplaces (verite terrain = OFFLOAD)

In [ ]:
moved_acc = df[df["gt_decision"] == "OFFLOAD"].groupby(["approach", "model_short"])["moved_correct"].mean().unstack("approach")
display(moved_acc.reindex(index=MODEL_ORDER)[APPROACH_ORDER].round(3))


## Taux de reponse exploitable (parsing reussi -- 1.0 par construction pour le pipeline)

In [ ]:
parse_rate = df.groupby(["approach", "model_short"])["parsed_ok"].mean().unstack("approach")
display(parse_rate.reindex(index=MODEL_ORDER)[APPROACH_ORDER].round(3))


## Figure -- exactitude de la decision, pipeline vs direct vs cot, par modele

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR = ROOT / "eval" / "comparison_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def plot_comparison(metric_df, title, filename, ylabel=None, ylim=(0, 1), legend_loc="upper right"):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    plot_df = metric_df.reindex(index=MODEL_ORDER)[APPROACH_ORDER]
    plot_df.plot.bar(ax=ax, color=[APPROACH_COLORS[a] for a in APPROACH_ORDER], width=0.8, edgecolor="white", linewidth=0.6)
    ax.set_title(title)
    ax.set_ylabel(ylabel or title)
    ax.set_xlabel("")
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_xticklabels(plot_df.index, rotation=0)
    ax.legend([APPROACH_LABELS[a] for a in APPROACH_ORDER], loc=legend_loc, frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Sauvegarde : {FIGURES_DIR / filename}")


plot_comparison(decision_acc, "Exactitude de la decision (STAY/OFFLOAD/INFEASIBLE)", "decision_accuracy_comparison.png")

In [ ]:
plot_comparison(real_valid, "Validite reelle du placement final (hors INFEASIBLE)", "real_valid_comparison.png")


In [ ]:
plot_comparison(moved_acc, "Exactitude des services deplaces (si verite terrain = OFFLOAD)", "moved_accuracy_comparison.png")


## Chargement -- decision de REFERENCE (necessity_checker + decision_engine sur les VRAIES exigences)

Voir eval/run_reference_decision.py -- necessity_checker/decision_engine
tournent directement sur `ground_truth.requirements`, sans aucun LLM. C'est
le nombre de migrations qu'il faudrait faire si l'extraction etait parfaite --
l'optimum reel, independant du modele et de la strategie. Deja verifie
coherent avec `ground_truth.decision`/`moved_components` a 99,2% pres (le
reste vient d'un arrondi `floor()` connu sur les capacites de noeuds, pas
d'une erreur de methode).

In [ ]:
REFERENCE_PATH = ROOT / "eval" / "results_reference" / "reference_decisions.jsonl"

ref_rows = []
with open(REFERENCE_PATH) as fh:
    for line in fh:
        r = json.loads(line)
        ref_rows.append({"scenario_id": r["scenario_id"], "n_moved_ref": len(r["ref_moved"])})

df_ref = pd.DataFrame(ref_rows)
ref_mean_migrations = df_ref["n_moved_ref"].mean()
print(f"Migrations moyennes, reference (optimum reel, 500 scenarios) : {ref_mean_migrations:.3f}")


## Optimalite -- nombre moyen de migrations par scenario (statistique DESCRIPTIVE seulement)

**Cette moyenne brute ne doit pas servir seule a comparer les approches** --
elle melange des causes opposees (faux STAY qui comptent 0 migration alors
qu'il en fallait, sur-migration qui gonfle le compte) dans un seul chiffre,
qui peut artificiellement se rapprocher de la reference par pure
coincidence. La vraie comparaison (taux de faisabilite, taux d'optimalite
parmi les solutions faisables, exces de migration) suit plus bas.

In [ ]:
migrations_by_approach = df.dropna(subset=["n_moved_pred"]).groupby(["approach", "model_short"])["n_moved_pred"].mean().unstack("approach")
migrations_by_approach = migrations_by_approach.reindex(index=MODEL_ORDER)[APPROACH_ORDER]
migrations_by_approach["reference (optimum)"] = ref_mean_migrations
display(migrations_by_approach.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
plot_df = migrations_by_approach[APPROACH_ORDER]
plot_df.plot.bar(ax=ax, color=[APPROACH_COLORS[a] for a in APPROACH_ORDER], width=0.8, edgecolor="white", linewidth=0.6)
ax.axhline(ref_mean_migrations, color="#333333", linestyle="--", linewidth=1.5, label="Reference (optimum reel)")
ax.set_title("Nombre moyen de migrations par scenario")
ax.set_ylabel("Migrations moyennes")
ax.set_xlabel("")
ax.set_xticklabels(plot_df.index, rotation=0)
ax.legend([APPROACH_LABELS[a] for a in APPROACH_ORDER] + ["Reference (optimum reel)"], loc="upper right", frameon=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "migrations_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'migrations_comparison.png'}")


## Cause racine des faux STAY (pipeline) -- extraction, pas necessity_checker

Verification : pour les 261 scenarios ou le pipeline predit STAY alors que
la verite terrain dit OFFLOAD, on rejoue `necessity_checker_node` en lui
donnant les VRAIES exigences du scenario (`ground_truth.requirements`) a la
place de celles extraites par le LLM -- meme infrastructure que
`eval/run_reference_decision.py`.

**Resultat : 261/261 (100%) passent a "reconfiguration_required" des qu'on
leur donne les vraies exigences.** `necessity_checker` lui-meme n'a jamais
tort -- le faux STAY est entierement un symptome de l'extraction en amont
(exigence manquee ou rejetee par `validate_and_normalize`), jamais un bug
de `necessity_checker`.

## Methodologie rigoureuse -- 3 indicateurs distincts, jamais un seul chiffre

1. **Decision accuracy** -- la bonne decision parmi STAY/OFFLOAD/INFEASIBLE est-elle prise ?
2. **OFFLOAD recall** -- parmi les scenarios ou une reconfiguration est reellement necessaire,
   quelle proportion l'approche declenche-t-elle (peu importe si le placement propose est bon) ?
3. **GT feasibility rate** -- le placement retourne (ou le placement courant, pour un STAY)
   satisfait-il REELLEMENT les vraies exigences -- deja calcule plus haut (`real_valid`),
   exclut INFEASIBLE par construction.

Puis, UNIQUEMENT sur les scenarios OFFLOAD et UNIQUEMENT quand le placement
propose est faisable : l'ecart de migration `delta_M = n_moved_pred - n_moved_ref`
(jamais negatif si la reference est bien optimale -- verifie plus tot a 99,2% pres).

In [ ]:
# OFFLOAD recall -- parmi gt_decision == OFFLOAD, l'approche a-t-elle bien dit OFFLOAD
# (independamment de la qualite du placement propose) ?
df["offload_recall_flag"] = df.apply(
    lambda row: (row["pred_decision"] == "OFFLOAD") if row["gt_decision"] == "OFFLOAD" else None, axis=1
)

# Merge la reference (n_moved_ref, par scenario_id, independant de l'approche/modele).
ref_lookup = df_ref.set_index("scenario_id")["n_moved_ref"]
df["n_moved_ref"] = df["scenario_id"].map(ref_lookup)
df["delta_m"] = df["n_moved_pred"] - df["n_moved_ref"]


def classify(row):
    """4 categories, seulement definies quand gt_decision == OFFLOAD (une
    reconfiguration est reellement necessaire) -- voir la methodologie
    ci-dessus."""
    if row["gt_decision"] != "OFFLOAD":
        return None
    if not row["parsed_ok"]:
        return "no_feasible_solution_returned"
    if row["pred_decision"] == "INFEASIBLE":
        return "no_feasible_solution_returned"
    if not row["real_valid"]:
        return "infeasible_placement"  # inclut le faux STAY (placement courant garde, mais pas faisable)
    if row["delta_m"] == 0:
        return "optimal"
    return "over_migration"


df["migration_category"] = df.apply(classify, axis=1)
df["migration_category"].value_counts()


### Table 1 -- qualite de la decision (tous scenarios)

In [ ]:
table1 = pd.DataFrame({
    "decision_accuracy": df.groupby("approach")["decision_correct"].mean(),
    "offload_recall": df.groupby("approach")["offload_recall_flag"].mean(),
    "gt_feasibility_rate": df[df["gt_decision"] != "INFEASIBLE"].groupby("approach")["real_valid"].mean(),
}).reindex(APPROACH_ORDER)
table1.loc["reference (oracle)"] = [1.0, 1.0, 1.0]  # par construction : optimum reel, toujours correct/faisable
display(table1.round(3))


### Table 1 (detail) -- qualite de la decision, par approche x modele x strategie

In [ ]:
table1_detail = pd.DataFrame({
    "decision_accuracy": df.groupby(["approach", "model_short", "strategy"])["decision_correct"].mean(),
    "offload_recall": df.groupby(["approach", "model_short", "strategy"])["offload_recall_flag"].mean(),
    "gt_feasibility_rate": df[df["gt_decision"] != "INFEASIBLE"].groupby(["approach", "model_short", "strategy"])["real_valid"].mean(),
}).reindex(pd.MultiIndex.from_product([APPROACH_ORDER, MODEL_ORDER, STRATEGY_ORDER], names=["approach", "model_short", "strategy"]))
display(table1_detail.round(3))


### Table 2 -- qualite d'optimisation, UNIQUEMENT sur les scenarios ou une reconfiguration est reellement necessaire (ground truth = OFFLOAD)

`optimal_rate_among_feasible` et `mean_excess_migrations_among_feasible`
sont calcules SEULEMENT parmi les placements faisables -- le taux de
faisabilite est donc indispensable pour les interpreter (un taux
d'optimalite de 100% sur 5% de cas faisables ne veut pas dire grand-chose).

In [ ]:
offload_df = df[df["gt_decision"] == "OFFLOAD"]
feasible_df = offload_df[offload_df["real_valid"]]

table2 = pd.DataFrame({
    "feasible_rate": offload_df.groupby("approach")["real_valid"].mean(),
    "optimal_rate_among_feasible": feasible_df.groupby("approach")["delta_m"].apply(lambda s: (s == 0).mean()),
    "mean_excess_migrations_among_feasible": feasible_df.groupby("approach")["delta_m"].mean(),
}).reindex(APPROACH_ORDER)
table2.loc["reference (oracle)"] = [1.0, 1.0, 0.0]
display(table2.round(3))


### Table 2 (detail) -- qualite d'optimisation, par approche x modele x strategie (scenarios OFFLOAD seulement)

In [ ]:
table2_detail = pd.DataFrame({
    "feasible_rate": offload_df.groupby(["approach", "model_short", "strategy"])["real_valid"].mean(),
    "optimal_rate_among_feasible": feasible_df.groupby(["approach", "model_short", "strategy"])["delta_m"].apply(lambda s: (s == 0).mean()),
    "mean_excess_migrations_among_feasible": feasible_df.groupby(["approach", "model_short", "strategy"])["delta_m"].mean(),
}).reindex(pd.MultiIndex.from_product([APPROACH_ORDER, MODEL_ORDER, STRATEGY_ORDER], names=["approach", "model_short", "strategy"]))
display(table2_detail.round(3))


### Figure -- repartition en 4 categories, par approche (scenarios OFFLOAD seulement), barres empilees a 100%

In [ ]:
CATEGORY_ORDER = ["optimal", "over_migration", "infeasible_placement", "no_feasible_solution_returned"]
CATEGORY_LABELS = {
    "optimal": "Optimal", "over_migration": "Sur-migration",
    "infeasible_placement": "Placement infaisable (dont faux STAY)",
    "no_feasible_solution_returned": "Aucune solution retournee",
}
CATEGORY_COLORS = {"optimal": "#28A69A", "over_migration": "#3569A8", "infeasible_placement": "#E99A43", "no_feasible_solution_returned": "#B23A48"}

cat_counts = offload_df.groupby(["approach", "migration_category"]).size().unstack("migration_category")
cat_pct = cat_counts.div(cat_counts.sum(axis=1), axis=0).reindex(index=APPROACH_ORDER)[CATEGORY_ORDER]

fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = pd.Series(0.0, index=cat_pct.index)
for cat in CATEGORY_ORDER:
    ax.bar([APPROACH_LABELS[a] for a in cat_pct.index], cat_pct[cat], bottom=bottom,
           label=CATEGORY_LABELS[cat], color=CATEGORY_COLORS[cat], edgecolor="white", linewidth=0.6)
    bottom += cat_pct[cat]
ax.set_title("Repartition des scenarios OFFLOAD par resultat (toutes modeles/strategies confondus)")
ax.set_ylabel("Proportion des scenarios OFFLOAD")
ax.set_ylim(0, 1)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "migration_categories_stacked.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'migration_categories_stacked.png'}")


## Cout -- temps et tokens, par approche x modele

`elapsed_s` : pipeline/ablation font DEUX appels LLM par scenario
(extraction + explanation, voir `total_elapsed_s`) ; direct/cot n'en font
qu'UN (voir `elapsed_s` du record). `total_tokens` : somme input+output
(+ reasoning cache, deja inclus par OpenRouter) des memes appels.

In [ ]:
cost_summary = df.groupby(["approach", "model_short"]).agg(
    mean_elapsed_s=("elapsed_s", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
)
display(cost_summary.reindex(
    index=pd.MultiIndex.from_product([APPROACH_ORDER, MODEL_ORDER])
).round(2))


In [ ]:
time_by_approach = df.groupby(["approach", "model_short"])["elapsed_s"].mean().unstack("approach").reindex(index=MODEL_ORDER)[APPROACH_ORDER]
plot_comparison(time_by_approach, "Temps moyen par scenario (s)", "elapsed_s_comparison.png",
                 ylabel="secondes", ylim=None, legend_loc="upper left")


In [ ]:
tokens_by_approach = df.groupby(["approach", "model_short"])["total_tokens"].mean().unstack("approach").reindex(index=MODEL_ORDER)[APPROACH_ORDER]
plot_comparison(tokens_by_approach, "Tokens moyens par scenario", "tokens_comparison.png",
                 ylabel="tokens (entree+sortie)", ylim=None, legend_loc="upper left")


## Livrables pour la section Experimental Results

Clarifications methodologiques avant les tableaux/figures finaux (point de
vigilance souleve) :

1. **Denominateurs des metriques de faisabilite** -- deux denominateurs
   differents, par design, pas une incoherence :
   - `gt_feasibility_rate` (Table 1) : denominateur = scenarios STAY + OFFLOAD
     (tous sauf INFEASIBLE). Mesure la qualite generale du placement final.
   - `feasible_rate` (Table 2) : denominateur = scenarios OFFLOAD seulement.
     Mesure specifiquement la qualite de la RECHERCHE de placement quand une
     reconfiguration est reellement necessaire.
2. **Traitement des scenarios INFEASIBLE** : exclus des deux metriques de
   faisabilite ci-dessus (aucun placement admissible n'existe, le taux
   serait toujours 0% par construction -- non informatif), mais INCLUS dans
   `decision_accuracy` (predire INFEASIBLE correctement reste un signal de
   qualite de decision).
3. **"Optimal" quand le solveur ne prouve pas l'optimalite** : verifie sur
   les 15500 appels CP-SAT de tout le projet (reference + 2 campagnes
   completes) -- le statut n'a JAMAIS ete autre chose que `OPTIMAL` ou
   `INFEASIBLE`. `decision_engine_cp_sat.py` leve meme une exception si un
   statut `FEASIBLE` non prouve survenait (voir son code, lignes 92-104) --
   cela ne s'est jamais produit. Chaque reference utilisee est donc une
   preuve d'optimalite, jamais une estimation.

Change `MAIN_MODEL`/`MAIN_STRATEGY` ci-dessous pour choisir l'exemple a
mettre en avant dans le papier -- le reste des tableaux/figures de ce
notebook reste disponible pour toute autre combinaison.

In [ ]:
MAIN_MODEL = "mistral-small-3.2-24b"  # <- changer ici pour choisir l'exemple du papier
MAIN_STRATEGY = "few_shot"             # <- idem

print(f"Exemple retenu : {MAIN_MODEL} / {MAIN_STRATEGY}")


### 1. Tableau principal -- Pipeline / Ablation / Direct LLM / CoT, a modele et strategie fixes

Une seule table, toutes les metriques des Tables 1 et 2 reunies, pour cette
combinaison modele x strategie precise.

In [ ]:
decision_cols = table1_detail.xs((MAIN_MODEL, MAIN_STRATEGY), level=("model_short", "strategy"))
optim_cols = table2_detail.xs((MAIN_MODEL, MAIN_STRATEGY), level=("model_short", "strategy"))

main_table = decision_cols.join(optim_cols)
main_table.loc["reference (oracle)"] = (
    list(table1.loc["reference (oracle)"]) + list(table2.loc["reference (oracle)"])
)
display(main_table.round(3))


### 2. Figure en barres groupees -- Decision Accuracy et Ground-Truth Feasibility, 5 modeles, strategie fixee (`MAIN_STRATEGY`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)

for ax, col, title in [
    (axes[0], "decision_correct", "Decision Accuracy"),
    (axes[1], "real_valid", "Ground-Truth Feasibility Rate"),
]:
    if col == "real_valid":
        sub = df[(df["strategy"] == MAIN_STRATEGY) & (df["gt_decision"] != "INFEASIBLE")]
    else:
        sub = df[df["strategy"] == MAIN_STRATEGY]
    pivot = sub.groupby(["model_short", "approach"])[col].mean().unstack("approach")
    pivot = pivot.reindex(index=MODEL_ORDER)[APPROACH_ORDER]
    pivot.plot.bar(ax=ax, color=[APPROACH_COLORS[a] for a in APPROACH_ORDER], width=0.8, edgecolor="white", linewidth=0.6)
    ax.set_title(f"{title} -- strategie: {MAIN_STRATEGY}")
    ax.set_ylabel(title)
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.set_xticklabels(pivot.index, rotation=30, ha="right")
    ax.legend([APPROACH_LABELS[a] for a in APPROACH_ORDER], loc="upper right", frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(FIGURES_DIR / f"decision_and_feasibility_by_model_{MAIN_STRATEGY}.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / f'decision_and_feasibility_by_model_{MAIN_STRATEGY}.png'}")


### 3. Tableau de qualite d'optimisation

Deja produit plus haut (Table 2 agregee + Table 2 detail par modele x
strategie) -- rappele ici, filtre sur `MAIN_MODEL`/`MAIN_STRATEGY`, pour
rester a cote du tableau principal.

In [ ]:
display(optim_cols.round(3))


### 4. Analyse ciblee des faux STAY

Deja faite plus haut ("Cause racine des faux STAY") : sur les 261 faux STAY
du pipeline, **261/261 (100%) sont imputables a l'extraction** (exigence
manquee ou rejetee par `validate_and_normalize`), **0% a un bug de
`necessity_checker`** -- verifie en rejouant `necessity_checker_node` avec
les VRAIES exigences a la place de celles extraites.

Non refait separement pour `ablation` ici (meme mecanisme d'extraction que
`pipeline`, le taux de faux STAY y est meme plus eleve -- `offload_recall`
84,1% contre 91,2% pour le pipeline, voir Table 1), mais le test est
reproductible a l'identique en changeant simplement quel resultat on filtre
dans le script de verification.

## Conclusion chiffree

L'ecart entre le pipeline complet et les deux baselines single-call vient de
la DELEGATION du raisonnement combinatoire : le pipeline laisse CP-SAT
chercher et verifier le placement (toujours valide une fois trouve), alors
que direct/cot demandent au LLM de faire cette recherche lui-meme, en une
passe, sans aucune garantie de correction formelle.


In [ ]:
overall = pd.DataFrame({
    "decision_accuracy": df.groupby("approach")["decision_correct"].mean(),
    "real_valid (hors INFEASIBLE)": df[df["gt_decision"] != "INFEASIBLE"].groupby("approach")["real_valid"].mean(),
    "moved_accuracy (si OFFLOAD)": df[df["gt_decision"] == "OFFLOAD"].groupby("approach")["moved_correct"].mean(),
    "migrations_moyennes": df.dropna(subset=["n_moved_pred"]).groupby("approach")["n_moved_pred"].mean(),
    "mean_elapsed_s": df.groupby("approach")["elapsed_s"].mean(),
    "mean_total_tokens": df.groupby("approach")["total_tokens"].mean(),
}).reindex(APPROACH_ORDER)
overall.loc["reference (optimum)", "migrations_moyennes"] = ref_mean_migrations
display(overall.round(3))